[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/galvezam/eia-data-pipeline/blob/main/processing/natural_gas_crude_processing.ipynb)

# Natural Gas & Crude Oil Processing
**EIA Data Pipeline**

Datasets:
- `natural_gas_*.csv` — Monthly marketed production by state (MMcf)
- `natural_gas_trade_*.csv` — Annual state-level exports, imports, and interstate movements (MMcf)
- `crude_oil_imports_*.csv` — Monthly crude oil imports by country of origin → refinery state (thousand barrels)

Pipeline steps: **Clean → Transform → Analyze → Upload Parquet to S3**

## 0. Environment Setup & Configuration

In [1]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"

# Install Spark with Hadoop as the underlying scheduler
!wget -q https://downloads.apache.org/spark/spark-3.5.8/spark-3.5.8-bin-hadoop3.tgz
!tar xf spark-3.5.8-bin-hadoop3.tgz
!ls -l

os.environ["SPARK_HOME"] = "spark-3.5.8-bin-hadoop3"
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--packages org.apache.hadoop:hadoop-aws:3.3.4,"
    "com.amazonaws:aws-java-sdk-bundle:1.12.262 "
    "pyspark-shell"
)

!pip install -q findspark
import findspark
findspark.init()
print("Spark environment ready.")

total 398028
-rw-r--r--  1 root root   5594930 Apr  8 00:52 crude_oil_imports_20260401.csv
-rw-r--r--  1 root root    114184 Apr  8 00:52 natural_gas_20260401.csv
-rw-r--r--  1 root root    478914 Apr  8 00:52 natural_gas_trade_20260401.csv
drwxr-xr-x  1 root root      4096 Apr  2 13:31 sample_data
drwxr-xr-x 13 1001 1001      4096 Jan 12 04:31 spark-3.5.8-bin-hadoop3
-rw-r--r--  1 root root 401378872 Jan 12 04:34 spark-3.5.8-bin-hadoop3.tgz
Spark environment ready.


In [2]:
import sys

# AWS credentials — fill in here if using Section 5 (S3 upload)
AWS_ACCESS_KEY = ""
AWS_SECRET_KEY = ""
AWS_REGION     = ""
BUCKET_NAME    = ""

import glob

# After manually uploading files via the Colab sidebar (they land in /content/)
NAT_GAS_PATH   = glob.glob("natural_gas_*.csv")[0]
NG_TRADE_PATH  = glob.glob("natural_gas_trade_*.csv")[0]
CRUDE_PATH     = glob.glob("crude_oil_imports_*.csv")[0]

# ── S3 output prefixes ─────────────────────────────────────────────────────
S3_BASE = f"s3a://{BUCKET_NAME}/processed"

S3_NG_PRODUCTION   = f"{S3_BASE}/natural_gas_production/"
S3_NG_INTL_TRADE   = f"{S3_BASE}/natural_gas_trade_international/"
S3_NG_INTERSTATE   = f"{S3_BASE}/natural_gas_trade_interstate/"
S3_CRUDE_BY_ORIGIN = f"{S3_BASE}/crude_oil_imports_by_state_country/"
S3_CRUDE_STATE     = f"{S3_BASE}/crude_oil_imports_by_state/"
S3_CRUDE_GRADE     = f"{S3_BASE}/crude_oil_imports_grade_breakdown/"

print("Config loaded.")
print(f"  Natural gas:   {NAT_GAS_PATH}")
print(f"  NG trade:      {NG_TRADE_PATH}")
print(f"  Crude imports: {CRUDE_PATH}")
print(f"  S3 bucket:     {BUCKET_NAME}")

Config loaded.
  Natural gas:   natural_gas_20260401.csv
  NG trade:      natural_gas_trade_20260401.csv
  Crude imports: crude_oil_imports_20260401.csv
  S3 bucket:     cs4266-eia-big-data-bucket


## 1. Initialize Spark Session

In [3]:
from pyspark.sql import SparkSession

import os
print(os.environ.get('JAVA_HOME'))  # Should show the jdk-17 path


builder = (
    SparkSession.builder
    .appName('EIA Natural Gas & Crude Oil Processing')
    .config('spark.hadoop.validateOutputSpecs', False)
)

# Only wire up S3 if credentials are filled in — lets you run locally without AWS
if AWS_ACCESS_KEY and AWS_SECRET_KEY and AWS_REGION:
    builder = (
        builder
        .config('spark.hadoop.fs.s3a.impl',        'org.apache.hadoop.fs.s3a.S3AFileSystem')
        .config('spark.hadoop.fs.s3a.endpoint',    f's3.{AWS_REGION}.amazonaws.com')
        .config('spark.hadoop.fs.s3a.access.key',  AWS_ACCESS_KEY)
        .config('spark.hadoop.fs.s3a.secret.key',  AWS_SECRET_KEY)
    )
    print('S3 configured.')
else:
    print('No S3 credentials — local mode only (Sections 0–4).')

spark = builder.getOrCreate()
spark.sparkContext.setLogLevel('WARN')
print(f'Spark {spark.version} ready.')

/usr/lib/jvm/java-8-openjdk-amd64
S3 configured.
Spark 3.5.8 ready.


## 2. Load Raw Data

In [4]:
def read_csv(path):
    return (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .option("multiLine", True)   # handles quoted commas in city names
        .option("escape", '"')
        .csv(path)
    )

raw_ng    = read_csv(NAT_GAS_PATH)
raw_trade = read_csv(NG_TRADE_PATH)
raw_crude = read_csv(CRUDE_PATH)

print(f"Natural Gas Production raw rows : {raw_ng.count():>8,}")
print(f"Natural Gas Trade raw rows       : {raw_trade.count():>8,}")
print(f"Crude Oil Imports raw rows       : {raw_crude.count():>8,}")

Natural Gas Production raw rows :      887
Natural Gas Trade raw rows       :    2,815
Crude Oil Imports raw rows       :   48,627


In [5]:
print("=== Natural Gas Production Schema ===")
raw_ng.printSchema()

print("\n=== Natural Gas Trade Schema ===")
raw_trade.printSchema()

print("\n=== Crude Oil Imports Schema ===")
raw_crude.printSchema()

=== Natural Gas Production Schema ===
root
 |-- period: string (nullable = true)
 |-- duoarea: string (nullable = true)
 |-- area-name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- product-name: string (nullable = true)
 |-- process: string (nullable = true)
 |-- process-name: string (nullable = true)
 |-- series: string (nullable = true)
 |-- series-description: string (nullable = true)
 |-- value: string (nullable = true)
 |-- units: string (nullable = true)


=== Natural Gas Trade Schema ===
root
 |-- period: string (nullable = true)
 |-- duoarea: string (nullable = true)
 |-- area-name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- product-name: string (nullable = true)
 |-- process: string (nullable = true)
 |-- process-name: string (nullable = true)
 |-- series: string (nullable = true)
 |-- series-description: string (nullable = true)
 |-- value: string (nullable = true)
 |-- units: string (nullable = true)


=== Crude Oil Imports S

## 3. Cleaning

### 3a. Natural Gas Production

In [6]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

ng_clean = (
    raw_ng
    # ── Remove rows with no reported production value ──────────────────────
    .filter(F.col("value").isNotNull())
    # ── Keep state-level rows only (duoarea starts with "S") ───────────────
    # Drops U.S. total (NUS), federal offshore (R3FM), and non-state entries
    .filter(F.col("duoarea").startswith("S"))
    # ── Keep only Marketed Production series (the primary production metric)─
    .filter(F.col("process") == "VGM")
    # ── Parse period ("YYYY-MM") to a proper date (first of month) ─────────
    .withColumn("period", F.to_date(F.col("period"), "yyyy-MM"))
    # ── Cast value to double ───────────────────────────────────────────────
    .withColumn("production_mmcf", F.col("value").cast("double"))
    # ── Rename and select relevant columns ─────────────────────────────────
    .withColumnRenamed("area-name", "state_name")
    .select("period", "duoarea", "state_name", "production_mmcf")
    # ── Drop duplicates that may arise from append-style file generation ───
    .dropDuplicates()
)

print(f"Natural Gas Production cleaned rows: {ng_clean.count():,}")
ng_clean.show(5)

Natural Gas Production cleaned rows: 192
+----------+-------+----------+---------------+
|    period|duoarea|state_name|production_mmcf|
+----------+-------+----------+---------------+
|2025-12-01|    SND|    USA-ND|        97900.0|
|2025-08-01|    SAK|    USA-AK|        28593.0|
|2025-04-01|    SUT|    USA-UT|        27878.0|
|2025-01-01|    SAR|    USA-AR|        27605.0|
|2025-12-01|    STX|     TEXAS|      1117708.0|
+----------+-------+----------+---------------+
only showing top 5 rows



### 3b. Natural Gas Trade

In [7]:
# Map process codes to readable labels
PROCESS_LABELS = {
    "EEI": "Exports (Intl)",
    "IMI": "Imports (Intl)",
    "IMB": "Net Intl Movements",
    "MID": "Interstate Deliveries",
    "MIN": "Net Interstate Receipts",
    "MIE": "Net Intl + Interstate",
}
process_map = F.create_map(
    *[item for pair in [(F.lit(k), F.lit(v)) for k, v in PROCESS_LABELS.items()] for item in pair]
)

trade_clean = (
    raw_trade
    # ── Remove rows with null value ────────────────────────────────────────
    .filter(F.col("value").isNotNull())
    # ── Keep state-level duoarea rows only ─────────────────────────────────
    .filter(F.col("duoarea").rlike(r"^S[A-Z]{2}"))
    # ── Keep only the 6 process codes of interest ──────────────────────────
    .filter(F.col("process").isin(list(PROCESS_LABELS.keys())))
    # ── Cast period to integer (annual data) ───────────────────────────────
    .withColumn("year", F.col("period").cast(IntegerType()))
    # ── Add human-readable process label ──────────────────────────────────
    .withColumn("process_label", process_map[F.col("process")])
    # ── Rename and select ──────────────────────────────────────────────────
    .withColumnRenamed("area-name", "state_name")
    .withColumn("value_mmcf", F.col("value").cast("double"))
    .select("year", "duoarea", "state_name", "process", "process_label", "value_mmcf", "units")
    .dropDuplicates()
)

print(f"Natural Gas Trade cleaned rows: {trade_clean.count():,}")
trade_clean.show(5)

Natural Gas Trade cleaned rows: 1,014
+----+-------+----------+-------+--------------------+----------+-----+
|year|duoarea|state_name|process|       process_label|value_mmcf|units|
+----+-------+----------+-------+--------------------+----------+-----+
|2024|SND-NCA|       CAN|    EEI|      Exports (Intl)|     237.0| MMCF|
|2024|SND-NCA|       CAN|    IMB|  Net Intl Movements|  473313.0| MMCF|
|2024|SWY-SUT|    USA-UT|    MID|Interstate Delive...|  930210.0| MMCF|
|2024|SLA-NCO|       COL|    EEI|      Exports (Intl)|   29408.0| MMCF|
|2024|SID-SNV|    USA-NV|    MID|Interstate Delive...|   26785.0| MMCF|
+----+-------+----------+-------+--------------------+----------+-----+
only showing top 5 rows



### 3c. Crude Oil Imports

In [8]:
crude_clean = (
    raw_crude
    # ── Remove rows with null quantity ─────────────────────────────────────
    .filter(F.col("quantity").isNotNull())
    # ── Filter to Refinery State level only to avoid double-counting ───────
    # The raw data repeats each import at 7 geographic levels:
    # Port PADD, Port State, Port, Refinery, Refinery PADD, Refinery State, US total.
    # We keep "Refinery State" as the most useful state-level grain.
    .filter(F.col("destinationTypeName") == "Refinery State")
    # ── Keep only actual country origins (not OPEC/Non-OPEC aggregates) ────
    .filter(F.col("originType") == "CTY")
    # ── Parse period to date ───────────────────────────────────────────────
    .withColumn("period", F.to_date(F.col("period"), "yyyy-MM"))
    # ── Cast quantity ──────────────────────────────────────────────────────
    .withColumn("quantity_thousand_bbl", F.col("quantity").cast("double"))
    # ── Rename destination column for clarity ──────────────────────────────
    .withColumnRenamed("destinationName", "refinery_state")
    .withColumnRenamed("originName",       "origin_country")
    .withColumnRenamed("gradeName",        "crude_grade")
    .select(
        "period", "origin_country", "originId",
        "refinery_state", "destinationId",
        "crude_grade", "quantity_thousand_bbl"
    )
    .dropDuplicates()
)

print(f"Crude Oil Imports cleaned rows: {crude_clean.count():,}")
crude_clean.show(5)

Crude Oil Imports cleaned rows: 1,287
+----------+--------------+--------+--------------+-------------+-----------+---------------------+
|    period|origin_country|originId|refinery_state|destinationId|crude_grade|quantity_thousand_bbl|
+----------+--------------+--------+--------------+-------------+-----------+---------------------+
|2025-01-01|        Canada|  CTY_CA|      Oklahoma|        RS_OK| Heavy Sour|              12619.0|
|2025-03-01|        Guyana|  CTY_GY|   Mississippi|        RS_MS|     Medium|               1496.0|
|2025-04-01|     Venezuela|  CTY_VE|     Louisiana|        RS_LA| Heavy Sour|               1444.0|
|2025-08-01|        Guyana|  CTY_GY|         Texas|        RS_TX|     Medium|                895.0|
|2025-09-01|        Canada|  CTY_CA|          Utah|        RS_UT|     Medium|                371.0|
+----------+--------------+--------+--------------+-------------+-----------+---------------------+
only showing top 5 rows



## 4. Analytics

### 4a. Natural Gas: State Production Over Time & % of US Total

In [9]:
# ── US monthly total (from cleaned state-level data) ──────────────────────
us_ng_total = (
    ng_clean
    .groupBy("period")
    .agg(F.sum("production_mmcf").alias("us_total_mmcf"))
)

# ── Join state totals with US total, compute % share ──────────────────────
ng_state_production = (
    ng_clean
    .join(us_ng_total, on="period", how="left")
    .withColumn(
        "pct_us_production",
        F.round(F.col("production_mmcf") / F.col("us_total_mmcf") * 100, 4)
    )
    .select(
        "period", "duoarea", "state_name",
        "production_mmcf", "us_total_mmcf", "pct_us_production"
    )
    .orderBy("period", F.col("production_mmcf").desc())
)

print(f"State production rows: {ng_state_production.count():,}")
ng_state_production.show(10)

State production rows: 192
+----------+-------+----------+---------------+-------------+-----------------+
|    period|duoarea|state_name|production_mmcf|us_total_mmcf|pct_us_production|
+----------+-------+----------+---------------+-------------+-----------------+
|2025-01-01|    STX|     TEXAS|      1049775.0|    3447813.0|          30.4476|
|2025-01-01|    SPA|    USA-PA|       661844.0|    3447813.0|          19.1961|
|2025-01-01|    SNM|    USA-NM|       318995.0|    3447813.0|           9.2521|
|2025-01-01|    SLA|    USA-LA|       285123.0|    3447813.0|           8.2697|
|2025-01-01|    SWV|    USA-WV|       283903.0|    3447813.0|           8.2343|
|2025-01-01|    SOK|    USA-OK|       230372.0|    3447813.0|           6.6817|
|2025-01-01|    SOH|      OHIO|       172182.0|    3447813.0|           4.9939|
|2025-01-01|    SCO|  COLORADO|       156067.0|    3447813.0|           4.5266|
|2025-01-01|    SND|    USA-ND|        98224.0|    3447813.0|           2.8489|
|2025-01-01| 

### 4b. Natural Gas: International Trade Summary by State (Annual)

In [10]:
intl_processes = ["EEI", "IMI", "IMB"]

ng_intl_trade = (
    trade_clean
    .filter(F.col("process").isin(intl_processes))
    .groupBy("year", "duoarea", "state_name", "process", "process_label")
    .agg(F.sum("value_mmcf").alias("total_mmcf"))
    .orderBy("year", "state_name", "process")
)

# ── Pivot wider: one row per state-year, columns for each process ──────────
ng_intl_trade_wide = (
    ng_intl_trade
    .groupBy("year", "duoarea", "state_name")
    .pivot("process", intl_processes)
    .agg(F.first("total_mmcf"))
    .withColumnRenamed("EEI", "exports_mmcf")
    .withColumnRenamed("IMI", "imports_mmcf")
    .withColumnRenamed("IMB", "net_intl_mmcf")
    .orderBy("year", "state_name")
)

print(f"International trade rows (wide): {ng_intl_trade_wide.count():,}")
ng_intl_trade_wide.show(10)

International trade rows (wide): 186
+----+-------+----------+------------+------------+-------------+
|year|duoarea|state_name|exports_mmcf|imports_mmcf|net_intl_mmcf|
+----+-------+----------+------------+------------+-------------+
|2024|SLA-NTC|       ARE|        NULL|         0.0|      -3064.0|
|2024|STX-NAR|       ARG|     22139.0|        NULL|     -22139.0|
|2024|SLA-NAR|       ARG|     26195.0|        NULL|     -26195.0|
|2024|SMD-NAR|       ARG|      3505.0|        NULL|      -3505.0|
|2024|SLA-NBE|       BEL|     30880.0|        NULL|         NULL|
|2024|STX-NBE|       BEL|      9960.0|        NULL|         NULL|
|2024|SLA-NBG|       BGD|     18959.0|        NULL|     -18959.0|
|2024|STX-NBG|       BGD|     21216.0|        NULL|     -21216.0|
|2024|SFL-NBF|       BHS|       496.0|        NULL|       -496.0|
|2024|STX-NBR|       BRA|     35483.0|        NULL|     -35483.0|
+----+-------+----------+------------+------------+-------------+
only showing top 10 rows



### 4c. Natural Gas: Interstate Movements by State (Annual)

In [11]:
interstate_processes = ["MID", "MIN"]

ng_interstate = (
    trade_clean
    .filter(F.col("process").isin(interstate_processes))
    .groupBy("year", "duoarea", "state_name", "process", "process_label")
    .agg(F.sum("value_mmcf").alias("total_mmcf"))
    .orderBy("year", "state_name", "process")
)

ng_interstate_wide = (
    ng_interstate
    .groupBy("year", "duoarea", "state_name")
    .pivot("process", interstate_processes)
    .agg(F.first("total_mmcf"))
    .withColumnRenamed("MID", "interstate_delivered_mmcf")
    .withColumnRenamed("MIN", "net_interstate_received_mmcf")
    .orderBy("year", "state_name")
)

print(f"Interstate movements rows (wide): {ng_interstate_wide.count():,}")
ng_interstate_wide.show(10)

Interstate movements rows (wide): 275
+----+-------+----------+-------------------------+----------------------------+
|year|duoarea|state_name|interstate_delivered_mmcf|net_interstate_received_mmcf|
+----+-------+----------+-------------------------+----------------------------+
|2024|SAZ-SCA|CALIFORNIA|                 898052.0|                   -816932.0|
|2024|SOR-SCA|CALIFORNIA|                 686167.0|                   -686167.0|
|2024|SCA-Z0S|CALIFORNIA|                 129533.0|                   1913274.0|
|2024|SNV-SCA|CALIFORNIA|                 458589.0|                   -410176.0|
|2024|SKS-SCO|  COLORADO|                   1472.0|                    170410.0|
|2024|SOK-SCO|  COLORADO|                      0.0|                     72776.0|
|2024|SNM-SCO|  COLORADO|                      0.0|                    289444.0|
|2024|SNE-SCO|  COLORADO|                 378460.0|                     29696.0|
|2024|SWY-SCO|  COLORADO|                 308412.0|                    

### 4d. Crude Oil: State Import Volume by Country of Origin (Monthly)

In [12]:
crude_by_origin = (
    crude_clean
    .groupBy("period", "refinery_state", "origin_country")
    .agg(F.sum("quantity_thousand_bbl").alias("total_thousand_bbl"))
    .orderBy("period", "refinery_state", F.col("total_thousand_bbl").desc())
)

print(f"Crude imports by state & origin rows: {crude_by_origin.count():,}")
crude_by_origin.show(10)

Crude imports by state & origin rows: 849
+----------+--------------+--------------+------------------+
|    period|refinery_state|origin_country|total_thousand_bbl|
+----------+--------------+--------------+------------------+
|2025-01-01|       Alabama|        Mexico|             495.0|
|2025-01-01|       Alabama|        Canada|             441.0|
|2025-01-01|        Alaska|        Canada|             471.0|
|2025-01-01|    California|          Iraq|            4883.0|
|2025-01-01|    California|        Canada|            4074.0|
|2025-01-01|    California|        Guyana|            3806.0|
|2025-01-01|    California|        Brazil|            3693.0|
|2025-01-01|    California|       Ecuador|            3509.0|
|2025-01-01|    California|  Saudi Arabia|            2478.0|
|2025-01-01|    California|      Colombia|            1950.0|
+----------+--------------+--------------+------------------+
only showing top 10 rows



### 4e. Crude Oil: Total State Imports Over Time & % of US Total

In [13]:
# ── US monthly total ───────────────────────────────────────────────────────
us_crude_total = (
    crude_clean
    .groupBy("period")
    .agg(F.sum("quantity_thousand_bbl").alias("us_total_thousand_bbl"))
)

# ── State totals + % of US ────────────────────────────────────────────────
crude_by_state = (
    crude_clean
    .groupBy("period", "refinery_state")
    .agg(F.sum("quantity_thousand_bbl").alias("total_thousand_bbl"))
    .join(us_crude_total, on="period", how="left")
    .withColumn(
        "pct_us_imports",
        F.round(F.col("total_thousand_bbl") / F.col("us_total_thousand_bbl") * 100, 4)
    )
    .orderBy("period", F.col("total_thousand_bbl").desc())
)

print(f"Crude imports by state rows: {crude_by_state.count():,}")
crude_by_state.show(10)

Crude imports by state rows: 286
+----------+--------------+------------------+---------------------+--------------+
|    period|refinery_state|total_thousand_bbl|us_total_thousand_bbl|pct_us_imports|
+----------+--------------+------------------+---------------------+--------------+
|2025-01-01|      Illinois|           34855.0|             206130.0|       16.9092|
|2025-01-01|         Texas|           34386.0|             206130.0|       16.6817|
|2025-01-01|    California|           27530.0|             206130.0|       13.3556|
|2025-01-01|      Oklahoma|           13817.0|             206130.0|        6.7031|
|2025-01-01|    Washington|           12403.0|             206130.0|        6.0171|
|2025-01-01|       Indiana|           12082.0|             206130.0|        5.8613|
|2025-01-01|    New Jersey|           11221.0|             206130.0|        5.4437|
|2025-01-01|     Minnesota|           11099.0|             206130.0|        5.3845|
|2025-01-01|       Montana|            6442

### 4f. Crude Oil: Grade Breakdown per State (Monthly)

In [14]:
# ── State-month totals (for computing within-state grade share) ────────────
state_month_totals = (
    crude_clean
    .groupBy("period", "refinery_state")
    .agg(F.sum("quantity_thousand_bbl").alias("state_total_thousand_bbl"))
)

crude_grade_breakdown = (
    crude_clean
    .groupBy("period", "refinery_state", "crude_grade")
    .agg(F.sum("quantity_thousand_bbl").alias("grade_quantity_thousand_bbl"))
    .join(state_month_totals, on=["period", "refinery_state"], how="left")
    .withColumn(
        "pct_of_state_imports",
        F.round(F.col("grade_quantity_thousand_bbl") / F.col("state_total_thousand_bbl") * 100, 4)
    )
    .orderBy("period", "refinery_state", F.col("grade_quantity_thousand_bbl").desc())
)

print(f"Grade breakdown rows: {crude_grade_breakdown.count():,}")
crude_grade_breakdown.show(10)

Grade breakdown rows: 758
+----------+--------------+-----------+---------------------------+------------------------+--------------------+
|    period|refinery_state|crude_grade|grade_quantity_thousand_bbl|state_total_thousand_bbl|pct_of_state_imports|
+----------+--------------+-----------+---------------------------+------------------------+--------------------+
|2025-01-01|       Alabama| Heavy Sour|                      936.0|                   936.0|               100.0|
|2025-01-01|        Alaska|     Medium|                      471.0|                   471.0|               100.0|
|2025-01-01|    California|     Medium|                    14747.0|                 27530.0|              53.567|
|2025-01-01|    California| Heavy Sour|                     9956.0|                 27530.0|             36.1642|
|2025-01-01|    California|Light Sweet|                     1624.0|                 27530.0|               5.899|
|2025-01-01|    California| Light Sour|                     12

## 5. Write Parquet to S3

**Requires a configured S3 bucket.** Fill in `example_config.py` first.  
All cells above this point run entirely locally without AWS credentials.

In [15]:
def write_parquet(df, path, label):
    """Write a DataFrame to S3 as snappy-compressed Parquet."""
    print(f"Writing {label} → {path} ...")
    df.write.mode("overwrite").option("compression", "snappy").parquet(path)
    print(f"  ✓ Done")

write_parquet(ng_state_production,   S3_NG_PRODUCTION,   "Natural Gas State Production")
write_parquet(ng_intl_trade_wide,    S3_NG_INTL_TRADE,   "Natural Gas International Trade")
write_parquet(ng_interstate_wide,    S3_NG_INTERSTATE,   "Natural Gas Interstate Movements")
write_parquet(crude_by_origin,       S3_CRUDE_BY_ORIGIN, "Crude Imports by State & Country")
write_parquet(crude_by_state,        S3_CRUDE_STATE,     "Crude Imports by State")
write_parquet(crude_grade_breakdown, S3_CRUDE_GRADE,     "Crude Imports Grade Breakdown")

print("\nAll datasets written to S3 successfully.")

Writing Natural Gas State Production → s3a://cs4266-eia-big-data-bucket/processed/natural_gas_production/ ...
  ✓ Done
Writing Natural Gas International Trade → s3a://cs4266-eia-big-data-bucket/processed/natural_gas_trade_international/ ...
  ✓ Done
Writing Natural Gas Interstate Movements → s3a://cs4266-eia-big-data-bucket/processed/natural_gas_trade_interstate/ ...
  ✓ Done
Writing Crude Imports by State & Country → s3a://cs4266-eia-big-data-bucket/processed/crude_oil_imports_by_state_country/ ...
  ✓ Done
Writing Crude Imports by State → s3a://cs4266-eia-big-data-bucket/processed/crude_oil_imports_by_state/ ...
  ✓ Done
Writing Crude Imports Grade Breakdown → s3a://cs4266-eia-big-data-bucket/processed/crude_oil_imports_grade_breakdown/ ...
  ✓ Done

All datasets written to S3 successfully.


## 6. Stop Spark

In [16]:
spark.stop()
print("Spark session stopped.")

Spark session stopped.
